In [1]:
import pandas as pd
import numpy as np
from pipeline.ingestion import load_data
from pipeline.features import build_features, FEATURE_COLS
from pipeline.score import score_transactions, score_mad, score_mcd, score_in_chunks, METRIC_FEATURES
from pipeline.classification import analyze_threshold, classify_transactions
from pipeline.explanation import run_explanation, prepare_attributes
from pipeline.evaluation import evaluate_pipeline, compare_scorers

/opt/anaconda3/envs/ml_hw4/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


# Feature Engineering

In [2]:
df = load_data("data/transactions_with_labels.csv")

# combine separate date/time fields into single datetime type
if "DateTime" not in df.columns:
    df["DateTime"] = pd.to_datetime(df["Date"] + " " + df["Time"])

# features are already present in the saved CSV — only build if missing
account_feature_cols = ["sender_fan_out_ratio", "sender_amount_cv", "receiver_fan_in_ratio"]
if not all(col in df.columns for col in account_feature_cols):
    from pipeline.explanation import prepare_attributes
    df = build_features(df)
    df = prepare_attributes(df)

print(f"Shape: {df.shape}")
print(df.columns.tolist())

Shape: (9504852, 37)
['Time', 'Date', 'Sender_account', 'Receiver_account', 'Amount', 'Payment_currency', 'Received_currency', 'Sender_bank_location', 'Receiver_bank_location', 'Payment_type', 'Is_laundering', 'Laundering_type', 'DateTime', 'log_amount', 'is_currency_conversion', 'sender_blacklist', 'sender_greylist', 'receiver_blacklist', 'receiver_greylist', 'sender_high_risk', 'receiver_high_risk', 'any_high_risk', 'just_below_2k', 'below_2k_margin', 'just_below_5k', 'below_5k_margin', 'just_below_10k', 'below_10k_margin', 'just_below_50k', 'below_50k_margin', 'is_off_hours', 'is_weekend', 'sender_fan_out_ratio', 'sender_amount_cv', 'sender_unique_receiver_countries', 'receiver_fan_in_ratio', 'time_since_last_tx_hours']


# Getting Rid of Laundering Columns

In [3]:
df_with_labels = df.copy()

df_without_labels = df.drop(
    columns=["Is_laundering", "Laundering_type"],
    errors="ignore"
).copy()

print("With labels:", df_with_labels.shape)
print("Without labels:", df_without_labels.shape)
# df_with_labels.to_csv("data/transactions_with_labels.csv", index=False)
# df_without_labels.to_csv("data/transactions_without_labels.csv", index=False)

With labels: (9504852, 37)
Without labels: (9504852, 35)


# Isolation Forest Scoring

In [4]:
SCORING_FEATURES = [
    "log_amount",
    "sender_fan_out_ratio",
    "sender_amount_cv",
    "receiver_fan_in_ratio",
    "time_since_last_tx_hours",
    "is_currency_conversion",
    "any_high_risk",
    "just_below_10k",
    "below_10k_margin",
    "sender_unique_receiver_countries",
]


scored_df, clf, scaler = score_transactions(df_without_labels, features=SCORING_FEATURES) # using an isolation forest

print(f"Shape: {scored_df.shape}")
print(f"Score range: [{scored_df['anomaly_score'].min():.4f}, {scored_df['anomaly_score'].max():.4f}]")
print("\nTop 10 Highest Risk Transactions (Isolation Forest):")
print(scored_df.nlargest(10, "anomaly_score")[["Sender_account", "Receiver_account", "Amount", "anomaly_score", "anomaly_score_normalized"]])

Training Isolation Forest on 9,504,852 transactions


[Parallel(n_jobs=10)]: Using backend ThreadingBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   2 out of  10 | elapsed:    1.5s remaining:    6.1s
[Parallel(n_jobs=10)]: Done  10 out of  10 | elapsed:    1.7s finished
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    9.5s
[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed:   19.5s finished
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    3.8s
[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed:    7.9s finished
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    9.5s
[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed:   19.4s finished
[Parallel(n_jobs=1)]: Done  49 tasks      | elapsed:    3.8s
[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed:    7.9s finished


Scoring complete. Score range: [0.3584, 0.7805]
Score percentiles:
 90th: 0.5047
 95th: 0.5381
 99th: 0.6167
 99.5th: 0.6494
 99.9th: 0.7020
Shape: (9504852, 37)
Score range: [0.3584, 0.7805]

Top 10 Highest Risk Transactions (Isolation Forest):
        Sender_account Receiver_account         Amount  anomaly_score  \
5746945     9401291265       1614632149    9849.139648       0.780476   
8349457     4735340495       3366423085    9643.599609       0.777120   
7281101     5732243739       4032397938  104290.851562       0.774215   
6666104     3588788631       9417119982    9049.549805       0.774049   
8321966     7094124432       3818650181  170955.406250       0.773474   
8013997     3620408520       6129366186  107224.476562       0.772645   
7139046     6692553022       1397069819  101382.789062       0.771600   
5049462     6598676049       2825539242    9003.049805       0.771352   
8333445     6485403942       1905312143    3115.590088       0.771309   
4406007     3593562426  

## MAD Scoring (Median Absolute Deviation)

In [5]:
scored_df, mad_params = score_mad(scored_df, features=SCORING_FEATURES)
print(f"MAD score range: [{scored_df['anomaly_score_mad'].min():.4f}, {scored_df['anomaly_score_mad'].max():.4f}]")
print("\nTop 10 Highest Risk Transactions (MAD):")
print(scored_df.nlargest(10, "anomaly_score_mad")[["Sender_account", "Receiver_account", "Amount", "anomaly_score_mad", "anomaly_score_mad_normalized"]])

MAD scoring complete. Score range: [0.1614, 353.9997]
  90th: 34.0576
  95th: 212.5136
  99th: 292.6707
  99.5th: 353.9997
  99.9th: 353.9997
MAD score range: [0.1614, 353.9997]

Top 10 Highest Risk Transactions (MAD):
        Sender_account Receiver_account        Amount  anomaly_score_mad  \
1522070     5611156657       3697911326    800.570007         353.999736   
1523951     5508348233       6647888917   6204.189941         353.999736   
1532171     3935240690       4835743431    717.219971         353.999736   
1535546     5637214862       3175190058   4738.290039         353.999736   
1536318     4668123872        905071615   1794.400024         353.999736   
1546605      908559902       1726662427  34029.671875         353.999736   
1557451     5371990142       2557737337   8702.940430         353.999736   
1558844      376736970       7092971860   2191.949951         353.999736   
1562427     9208494421       2813573977   1801.339966         353.999736   
1564864     750505364

## MCD Scoring (Minimum Covariance Determinant)

In [6]:
# Fitted on a 10k subsample (FastMCD is too expensive for 9.5M rows)
scored_df, mcd = score_mcd(scored_df, features=SCORING_FEATURES)

print(f"MCD score range: [{scored_df['anomaly_score_mcd'].min():.4f}, {scored_df['anomaly_score_mcd'].max():.4f}]")
print("\nTop 10 Highest Risk Transactions (MCD):")
print(scored_df.nlargest(10, "anomaly_score_mcd")[["Sender_account", "Receiver_account", "Amount", "anomaly_score_mcd", "anomaly_score_mcd_normalized"]])

Fitting MinCovDet on 100,000 sampled transactions...


/opt/anaconda3/envs/ml_hw4/lib/python3.11/site-packages/sklearn/covariance/_robust_covariance.py:793: UserWarning: The covariance matrix associated to your dataset is not full rank
  warnings.warn(


Scoring 9,504,852 transactions via Mahalanobis distance...
MCD scoring complete. Score range: [0.4174, 12540.0998]
  90th: 44.4907
  95th: 178.7588
  99th: 1901.7591
  99.5th: 1995.1601
  99.9th: 3931.7218
MCD score range: [0.4174, 12540.0998]

Top 10 Highest Risk Transactions (MCD):
        Sender_account Receiver_account         Amount  anomaly_score_mcd  \
8680877      733362131       6028013588  533580.562500       12540.099806   
9422579      431554764       3331273253   84511.320312       11980.435349   
8336896     9332541212       4129302823  108440.531250       11789.459695   
9053808     8024784004       1220597002   10314.650391       11773.233170   
9479404     4630999858       9309727487    3303.459961       11609.884706   
9285794     3256287270       1915242789   16002.480469       11573.627657   
9376189     7011771154       8619990433   28676.289062       11491.269751   
9208659      357128976       6305804357    6872.560059       11451.245441   
9191505     3919231535

## Scorer Comparison: Isolation Forest vs MAD vs MCD

In [7]:
# Side-by-side comparison of all three scorers at top 1% threshold
# Reattach labels temporarily for evaluation
scored_df_eval = scored_df.merge(
    df_with_labels[["Sender_account", "Receiver_account", "DateTime", "Is_laundering"]],
    on=["Sender_account", "Receiver_account", "DateTime"],
    how="left"
)

comparison = compare_scorers(scored_df_eval, label_col="Is_laundering", threshold_pct=1.0)
print("\nComparison DataFrame:")
print(comparison.to_string(index=False))


SCORER COMPARISON  (threshold = top 1.0%)
Scorer                  Flagged       TP  Precision     Recall       F1   ROC-AUC
----------------------------------------------------------------------
Isolation Forest         95,056    3,092      3.25%     31.32%   0.0589    0.8904
MAD                     372,840    1,546      0.41%     15.66%   0.0081    0.8565
MCD                      95,056    1,630      1.71%     16.51%   0.0311    0.8908

Comparison DataFrame:
          scorer         score_col  n_flagged  n_true_positives  precision   recall       f1  roc_auc
Isolation Forest     anomaly_score      95056              3092   0.032528 0.313177 0.058935 0.890444
             MAD anomaly_score_mad     372840              1546   0.004147 0.156589 0.008079 0.856493
             MCD anomaly_score_mcd      95056              1630   0.017148 0.165097 0.031069 0.890837


# Evaluation & Classification

In [8]:
# Classify transactions using optimal threshold (1% flagged)
classified_df, cutoff = classify_transactions(
    scored_df,
    score_col="anomaly_score",
    threshold_pct=1.0
)

print(f"Classification threshold: {cutoff:.4f}")
print(f"\nTop anomalous transactions (sample):")
print(classified_df[classified_df["is_anomalous"] == 1].nlargest(5, "anomaly_score")[
    ["Sender_account", "Receiver_account", "Amount", "anomaly_score", "is_anomalous"]
])

Classification at top 1.0%: 95,050 flagged as anomalous (1.000% of total)
Classification threshold: 0.6167

Top anomalous transactions (sample):
        Sender_account Receiver_account         Amount  anomaly_score  \
5746945     9401291265       1614632149    9849.139648       0.780476   
8349457     4735340495       3366423085    9643.599609       0.777120   
7281101     5732243739       4032397938  104290.851562       0.774215   
6666104     3588788631       9417119982    9049.549805       0.774049   
8321966     7094124432       3818650181  170955.406250       0.773474   

         is_anomalous  
5746945             1  
8349457             1  
7281101             1  
6666104             1  
8321966             1  


In [9]:
# Merge scored results with ground truth labels for evaluation
evaluation_df = classified_df.merge(
    df_with_labels[["Sender_account", "Receiver_account", "DateTime", "Is_laundering", "Laundering_type"]],
    on=["Sender_account", "Receiver_account", "DateTime"],
    how="left"
)

print(f"Evaluation dataset shape: {evaluation_df.shape}")
print(f"Ground truth labeling completeness: {evaluation_df['Is_laundering'].notna().sum()} / {len(evaluation_df)}")

# Analyze performance at different thresholds
print("\n" + "="*60)
print("Threshold Analysis")
print("="*60)
threshold_results = analyze_threshold(
    evaluation_df,
    score_col="anomaly_score",
    label_col="Is_laundering",
    thresholds=[0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
)

Evaluation dataset shape: (9505522, 44)
Ground truth labeling completeness: 9505522 / 9505522

Threshold Analysis
Top 0.1% flagged: 9,645 transactions
  True positives: 336 / 9,873 (3.4% recall)
  Precision: 3.48%
  F1: 0.0344

Top 0.5% flagged: 47,528 transactions
  True positives: 1,873 / 9,873 (19.0% recall)
  Precision: 3.94%
  F1: 0.0653

Top 1.0% flagged: 95,056 transactions
  True positives: 3,092 / 9,873 (31.3% recall)
  Precision: 3.25%
  F1: 0.0589

Top 2.0% flagged: 190,111 transactions
  True positives: 4,433 / 9,873 (44.9% recall)
  Precision: 2.33%
  F1: 0.0443

Top 5.0% flagged: 475,277 transactions
  True positives: 6,079 / 9,873 (61.6% recall)
  Precision: 1.28%
  F1: 0.0251

Top 10.0% flagged: 950,553 transactions
  True positives: 7,029 / 9,873 (71.2% recall)
  Precision: 0.74%
  F1: 0.0146

ROC-AUC: 0.8904


# Explanation: Discovering AML Patterns

In [10]:
# Use df_with_labels which has both anomaly scores and ground truth
explanation_df = df_with_labels.merge(
    classified_df[["Sender_account", "Receiver_account", "DateTime", "is_anomalous"]],
    on=["Sender_account", "Receiver_account", "DateTime"],
    how="left"
)

# Generate alerts ranking suspicious patterns by risk ratio
alerts = run_explanation(
    explanation_df,
    min_support=0.001,
    min_risk_ratio=3.0,
    max_combination_size=3,
    top_n=50
)
print('\n')
print("Top 20 suspicious pattern via Risk Ratios")
print(alerts.head(20).to_string())

Explanation stage: 95,050 outliers vs 9,505,522 total
Single-attribute candidates: 31
Encoding outlier transactions for FP-Growth:
Running FP-Growth on 33,448 outlier transactions, 31 candidate predicates
Found 170 frequent itemsets.


Top 20 suspicious pattern via Risk Ratios
                                                                                            predicates  outlier_count  outlier_support  pop_rate  risk_ratio  n_attributes
rank                                                                                                                                                                      
1                                             Payment_type=Cross-border ∧ Sender_bank_location=Morocco           2404         0.025293  0.001937   13.057247             2
2                                        Sender_bank_location=Morocco ∧ is_currency_conversion_str=yes           2398         0.025233  0.001933   13.051187             2
3            Payment_type=Cross-border

# Pipeline Evaluation Report

# Sample Output: Top 50 Alert, Annotated Against Known AML Typologies

In [11]:
# Full top-50 alert table sorted by risk ratio
print(f"Total alerts generated: {len(alerts)}")
print("\nTop 50 Alerts by Risk Ratio")
print("=" * 110)
display_cols = ["predicates", "n_attributes", "outlier_count", "outlier_support", "pop_rate", "risk_ratio"]
print(alerts[display_cols].head(50).to_string())

Total alerts generated: 50

Top 50 Alerts by Risk Ratio
                                                                                            predicates  n_attributes  outlier_count  outlier_support  pop_rate  risk_ratio
rank                                                                                                                                                                      
1                                             Payment_type=Cross-border ∧ Sender_bank_location=Morocco             2           2404         0.025293  0.001937   13.057247
2                                        Sender_bank_location=Morocco ∧ is_currency_conversion_str=yes             2           2398         0.025233  0.001933   13.051187
3            Payment_type=Cross-border ∧ Sender_bank_location=Morocco ∧ is_currency_conversion_str=yes             3           2398         0.025233  0.001933   13.051187
4     Payment_currency=Moroccan dirham ∧ Sender_bank_location=Morocco ∧ is_currency_conve

In [12]:
print(evaluation_df.columns.tolist())                                             


['Time', 'Date', 'Sender_account', 'Receiver_account', 'Amount', 'Payment_currency', 'Received_currency', 'Sender_bank_location', 'Receiver_bank_location', 'Payment_type', 'DateTime', 'log_amount', 'is_currency_conversion', 'sender_blacklist', 'sender_greylist', 'receiver_blacklist', 'receiver_greylist', 'sender_high_risk', 'receiver_high_risk', 'any_high_risk', 'just_below_2k', 'below_2k_margin', 'just_below_5k', 'below_5k_margin', 'just_below_10k', 'below_10k_margin', 'just_below_50k', 'below_50k_margin', 'is_off_hours', 'is_weekend', 'sender_fan_out_ratio', 'sender_amount_cv', 'sender_unique_receiver_countries', 'receiver_fan_in_ratio', 'time_since_last_tx_hours', 'anomaly_score', 'anomaly_score_normalized', 'anomaly_score_mad', 'anomaly_score_mad_normalized', 'anomaly_score_mcd', 'anomaly_score_mcd_normalized', 'is_anomalous', 'Is_laundering', 'Laundering_type']


In [13]:
def parse_predicate(predicate_str):
    """Parse 'col=val ∧ col=val ...' into a dict of {col: val}."""
    parts = predicate_str.split(" ∧ ")
    result = {}
    for part in parts:
        col, val = part.split("=", 1)
        result[col.strip()] = val.strip()
    return result

def filter_by_predicate(df, predicate_str):
    """Return rows of df matching all col=val constraints in predicate_str."""
    filters = parse_predicate(predicate_str)
    mask = pd.Series(True, index=df.index)
    for col, val in filters.items():
        if col in df.columns:
            mask &= df[col].astype(str) == val
    return df[mask]

# For the top 15 alerts: show Laundering_type distribution among matching flagged transactions
print("Typology breakdown for top 15 alerts")
print("Filters applied to evaluation_df (rows with is_anomalous == 1)\n")

flagged = evaluation_df[evaluation_df["is_anomalous"] == 1].copy()

for rank, row in alerts.head(15).iterrows():
    pred = row["predicates"]
    matched = filter_by_predicate(flagged, pred)
    total_matched = len(matched)
    laundering_in_matched = matched[matched["Is_laundering"] == 1]

    print(f"Rank {rank} | risk_ratio={row['risk_ratio']:.2f} | flagged matches: {total_matched:,} | "
          f"of which true suspicious: {len(laundering_in_matched):,} "
          f"({100*len(laundering_in_matched)/max(total_matched,1):.1f}%)")
    print(f"  Predicate: {pred}")

    if len(laundering_in_matched) > 0:
        typology_counts = laundering_in_matched["Laundering_type"].value_counts().head(5)
        for typology, count in typology_counts.items():
            print(f"    {typology}: {count}")
    else:
        print("    (no confirmed suspicious transactions in this alert segment)")
    print()

Typology breakdown for top 15 alerts
Filters applied to evaluation_df (rows with is_anomalous == 1)

Rank 1 | risk_ratio=13.06 | flagged matches: 846 | of which true suspicious: 28 (3.3%)
  Predicate: Payment_type=Cross-border ∧ Sender_bank_location=Morocco
    Structuring: 10
    Single_large: 7
    Cycle: 3
    Layered_Fan_In: 3
    Stacked Bipartite: 3

Rank 2 | risk_ratio=13.05 | flagged matches: 848 | of which true suspicious: 29 (3.4%)
  Predicate: Sender_bank_location=Morocco ∧ is_currency_conversion_str=yes
    Structuring: 11
    Single_large: 7
    Cycle: 3
    Layered_Fan_In: 3
    Stacked Bipartite: 3

Rank 3 | risk_ratio=13.05 | flagged matches: 846 | of which true suspicious: 28 (3.3%)
  Predicate: Payment_type=Cross-border ∧ Sender_bank_location=Morocco ∧ is_currency_conversion_str=yes
    Structuring: 10
    Single_large: 7
    Cycle: 3
    Layered_Fan_In: 3
    Stacked Bipartite: 3

Rank 4 | risk_ratio=12.95 | flagged matches: 832 | of which true suspicious: 28 (3.4%)


In [14]:
# Cross-reference: for each known suspicious typology, what fraction of its transactions
# are captured by at least one top-25 alert predicate?
top_predicates = alerts.head(25)["predicates"].tolist()

def matches_any_alert(row, predicates):
    for pred in predicates:
        filters = parse_predicate(pred)
        if all(str(row.get(col, "")) == val for col, val in filters.items()):
            return True
    return False

suspicious_only = evaluation_df[evaluation_df["Is_laundering"] == 1].copy()

# Check coverage per typology
print("Alert coverage by typology (% of true suspicious txns matching at least one top-25 alert)")
print("=" * 75)

typology_coverage = {}
for typology in suspicious_only["Laundering_type"].unique():
    subset = suspicious_only[suspicious_only["Laundering_type"] == typology]
    covered = subset.apply(lambda r: matches_any_alert(r, top_predicates), axis=1).sum()
    typology_coverage[typology] = (covered, len(subset))

for typology, (covered, total) in sorted(typology_coverage.items(), key=lambda x: -x[1][0]/max(x[1][1],1)):
    pct = 100 * covered / max(total, 1)
    bar = "█" * int(pct / 5)
    print(f"  {typology:<25} {covered:>4}/{total:<5} ({pct:5.1f}%) {bar}")

Alert coverage by typology (% of true suspicious txns matching at least one top-25 alert)
  Scatter-Gather              16/338   (  4.7%) 
  Cycle                       16/382   (  4.2%) 
  Bipartite                   15/383   (  3.9%) 
  Structuring                 68/1870  (  3.6%) 
  Single_large                 7/250   (  2.8%) 
  Fan_Out                      5/237   (  2.1%) 
  Layered_Fan_Out             10/529   (  1.9%) 
  Fan_In                       5/364   (  1.4%) 
  Behavioural_Change_2         4/345   (  1.2%) 
  Stacked Bipartite            5/506   (  1.0%) 
  Layered_Fan_In               6/656   (  0.9%) 
  Gather-Scatter               1/354   (  0.3%) 
  Smurfing                     0/932   (  0.0%) 
  Cash_Withdrawal              0/1334  (  0.0%) 
  Behavioural_Change_1         0/394   (  0.0%) 
  Over-Invoicing               0/54    (  0.0%) 
  Deposit-Send                 0/945   (  0.0%) 
